# DCGAN breve con MNIST

Este cuaderno genera dígitos sintéticos con una DCGAN. Se usan 1.000 imágenes de entrenamiento, redimensionadas a 32 × 32, y dos épocas para mantener la prueba ejecutable en CPU. La salida es exploratoria y se analiza con imágenes y pérdidas.

In [ ]:
import os, random, zipfile, numpy as np, torch, matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:",device)
transform=transforms.Compose([transforms.Resize(32),transforms.ToTensor(),transforms.Normalize((.5,),(.5,))])
full=MNIST("/content/data",train=True,download=True,transform=transform)
data=Subset(full,range(1000))
loader=DataLoader(data,batch_size=64,shuffle=True,drop_last=True,num_workers=0)
print("Imágenes:",len(data),"lotes/época:",len(loader))
real,_=next(iter(loader)); plt.figure(figsize=(7,3)); plt.axis("off"); plt.imshow(make_grid(real[:16],nrow=8,normalize=True).permute(1,2,0).squeeze(),cmap="gray"); plt.show()

In [ ]:
nz=64; width=32
class Generator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.ConvTranspose2d(nz,width*4,4,1,0,bias=False),nn.BatchNorm2d(width*4),nn.ReLU(True),nn.ConvTranspose2d(width*4,width*2,4,2,1,bias=False),nn.BatchNorm2d(width*2),nn.ReLU(True),nn.ConvTranspose2d(width*2,width,4,2,1,bias=False),nn.BatchNorm2d(width),nn.ReLU(True),nn.ConvTranspose2d(width,1,4,2,1,bias=False),nn.Tanh())
 def forward(self,z): return self.net(z)
class Discriminator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,width,4,2,1,bias=False),nn.LeakyReLU(.2,True),nn.Conv2d(width,width*2,4,2,1,bias=False),nn.BatchNorm2d(width*2),nn.LeakyReLU(.2,True),nn.Conv2d(width*2,width*4,4,2,1,bias=False),nn.BatchNorm2d(width*4),nn.LeakyReLU(.2,True),nn.Conv2d(width*4,1,4,1,0,bias=False),nn.Sigmoid())
 def forward(self,x): return self.net(x).view(-1)
G=Generator().to(device); D=Discriminator().to(device)
for module in list(G.modules())+list(D.modules()):
 if isinstance(module,(nn.Conv2d,nn.ConvTranspose2d,nn.BatchNorm2d)): nn.init.normal_(module.weight,0,.02)
criterion=nn.BCELoss(); optG=optim.Adam(G.parameters(),lr=.0002,betas=(.5,.999)); optD=optim.Adam(D.parameters(),lr=.0002,betas=(.5,.999)); fixed=torch.randn(64,nz,1,1,device=device)
print("Parámetros G:",sum(p.numel() for p in G.parameters()),"| D:",sum(p.numel() for p in D.parameters()))

In [ ]:
epochs=2; history={"G":[],"D":[]}; out="/content/resultados_dcgan"; os.makedirs(out,exist_ok=True)
for epoch in range(epochs):
 totalG=totalD=0.0
 for real,_ in loader:
  real=real.to(device); n=real.size(0); D.zero_grad(set_to_none=True)
  loss_real=criterion(D(real),torch.ones(n,device=device)); loss_real.backward()
  fake=G(torch.randn(n,nz,1,1,device=device)); loss_fake=criterion(D(fake.detach()),torch.zeros(n,device=device)); loss_fake.backward(); lossD=loss_real+loss_fake; optD.step()
  G.zero_grad(set_to_none=True); lossG=criterion(D(fake),torch.ones(n,device=device)); lossG.backward(); optG.step(); totalG+=lossG.item(); totalD+=lossD.item()
 history["G"].append(totalG/len(loader)); history["D"].append(totalD/len(loader))
 with torch.no_grad(): save_image(G(fixed).cpu(),f"{out}/epoca_{epoch+1}.png",normalize=True,nrow=8)
 print("Época %d/%d | pérdida D=%.4f | pérdida G=%.4f" % (epoch+1,epochs,history["D"][-1],history["G"][-1]))
plt.plot(history["D"],marker="o",label="Discriminador"); plt.plot(history["G"],marker="o",label="Generador"); plt.xlabel("Época"); plt.ylabel("Pérdida BCE"); plt.xticks(range(epochs),range(1,epochs+1)); plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.savefig(f"{out}/perdidas.png",dpi=160); plt.show()
torch.save({"G":G.state_dict(),"D":D.state_dict(),"history":history},f"{out}/dcgan_mnist.pt")
with zipfile.ZipFile("/content/evidencias_dcgan.zip","w",zipfile.ZIP_DEFLATED) as archive:
 for name in os.listdir(out): archive.write(f"{out}/{name}",arcname=name)
print("Archivos:",sorted(os.listdir(out)))

## Interpretación

Compare la legibilidad y variedad de ambas cuadrículas. Las pérdidas de una GAN no equivalen directamente a calidad visual; con un subconjunto pequeño y dos épocas, los dígitos pueden verse incompletos o repetidos. Registre esas limitaciones y no presente esta prueba como una generación de rostros.

In [ ]:
from google.colab import files
files.download("/content/evidencias_dcgan.zip")